# Tokenizer alignment — LAB09

Notebook này trả lời câu hỏi trung tâm của lab: annotation NER được gán theo
**word**, nhưng Transformer đọc **subword token**. Ta phải nhìn thấy `word_ids`,
`input_ids`, `labels` và quyết định cách căn chỉnh.

Mọi số liệu dưới đây được tính lại từ CoNLL-2003 test trong `data/` bằng
`bert-base-cased`.

In [1]:
import sys
from collections import Counter

import pandas as pd
from transformers import AutoTokenizer

from ner_utils import (
    IGNORE, ID2LABEL, LABELS, MODEL_DIR, bio_to_entities,
    align_tags, load_conll, read_json, tags_to_names,
)

pd.set_option("display.max_colwidth", 80)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
splits = load_conll()
print("tokenizer:", type(tokenizer).__name__, "| vocab:", tokenizer.vocab_size)
print({k: len(v) for k, v in splits.items()})

C:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tokenizer: BertTokenizer | vocab: 28996
{'train': 14041, 'validation': 3250, 'test': 3453}


## 1. Word -> subword

`is_split_into_words=True` giữ nguyên ranh giới word. `word_ids()` cho biết mỗi
subword thuộc word nào (`None` = special token).

In [2]:
words = ["Barack", "Obama", "visited", "New", "York", "."]
encoded = tokenizer(words, is_split_into_words=True, truncation=True, padding=True)
table = pd.DataFrame({
    "position": range(len(encoded["input_ids"])),
    "token": tokenizer.convert_ids_to_tokens(encoded["input_ids"]),
    "input_id": encoded["input_ids"],
    "attention_mask": encoded["attention_mask"],
    "word_id": encoded.word_ids(),
    "word": [words[w] if w is not None else "<special>" for w in encoded.word_ids()],
})
display(table)

,position,token,input_id,attention_mask,word_id,word
0,0,[CLS],101,1,NaN,<special>
1,1,Barack,14319,1,0.0,Barack
2,2,Obama,7661,1,1.0,Obama
3,3,visited,3891,1,2.0,visited
4,4,New,1203,1,3.0,New
5,5,York,1365,1,4.0,York
6,6,.,119,1,5.0,.
7,7,[SEP],102,1,NaN,<special>


## 2. Một từ thật sự bị chia trong corpus

Không phải từ nào cũng bị chia. Ví dụ dưới đây lấy từ test set: một entity word
bị WordPiece tách thành nhiều mảnh `##`.

In [3]:
example = None
for sentence in splits["test"]:
    enc = tokenizer(sentence["tokens"], is_split_into_words=True,
                    truncation=True, max_length=128)
    wids = enc.word_ids()
    for word_id in sorted({w for w in wids if w is not None}):
        if wids.count(word_id) > 1 and sentence["ner_tags"][word_id] != 0:
            example = (sentence, enc, word_id)
            break
    if example:
        break

sentence, enc, word_id = example
entity_at_word = {e.start: e for e in bio_to_entities(tags_to_names(sentence["ner_tags"]))}
rows = []
for i, (token, wid) in enumerate(zip(tokenizer.convert_ids_to_tokens(enc["input_ids"]),
                                     enc.word_ids())):
    rows.append({
        "token": token,
        "word_id": wid,
        "word": sentence["tokens"][wid] if wid is not None else "<special>",
        "gold_tag": ID2LABEL[sentence["ner_tags"][wid]] if wid is not None else "-",
    })
split_table = pd.DataFrame(rows)
print("sentence:", " ".join(sentence["tokens"]))
print("words:", sentence["tokens"])
print("split word:", sentence["tokens"][word_id],
      "->", tokenizer.tokenize(sentence["tokens"][word_id]))
display(split_table)

sentence: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
words: ['SOCCER', '-', 'JAPAN', 'GET', 'LUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISE', 'DEFEAT', '.']
split word: JAPAN -> ['J', '##AP', '##AN']


,token,word_id,word,gold_tag
0,[CLS],NaN,<special>,-
1,S,0.0,SOCCER,O
2,##OC,0.0,SOCCER,O
3,##CE,0.0,SOCCER,O
4,##R,0.0,SOCCER,O
5,-,1.0,-,O
6,J,2.0,JAPAN,B-LOC
7,##AP,2.0,JAPAN,B-LOC
8,##AN,2.0,JAPAN,B-LOC
9,GE,3.0,GET,O


## 3. Label alignment

Hai strategy của lab:

* **A — first subword**: `play` nhận `B-X`, `##ing` nhận `IGNORE` (-100).
* **B — propagate**: `play` nhận `B-X`, `##ing` nhận `I-X`.

`IGNORE` không có nghĩa token không tồn tại; nó chỉ bị loại khỏi loss:

$$L = -\sum_{i:\,y_i \ne IGNORE} \log P(y_i \mid x)$$

In [4]:
alignment_table = pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(enc["input_ids"]),
    "word_id": enc.word_ids(),
    "gold_word_tag": [
        ID2LABEL[sentence["ner_tags"][w]] if w is not None else "-"
        for w in enc.word_ids()
    ],
    "strategy_A": [ID2LABEL[l] if l != IGNORE else "IGNORE"
                   for l in align_tags(enc.word_ids(), sentence["ner_tags"], "first")],
    "strategy_B": [ID2LABEL[l] if l != IGNORE else "IGNORE"
                   for l in align_tags(enc.word_ids(), sentence["ner_tags"], "propagate")],
})
display(alignment_table)

n_ignore = sum(1 for l in align_tags(enc.word_ids(), sentence["ner_tags"], "first")
               if l == IGNORE)
print(f"IGNORE positions: {n_ignore} / {len(enc['input_ids'])} tokens in this sentence")
print("IGNORE index =", IGNORE)

,token,word_id,gold_word_tag,strategy_A,strategy_B
0,[CLS],NaN,-,IGNORE,IGNORE
1,S,0.0,O,O,O
2,##OC,0.0,O,IGNORE,O
3,##CE,0.0,O,IGNORE,O
4,##R,0.0,O,IGNORE,O
5,-,1.0,O,O,O
6,J,2.0,B-LOC,B-LOC,B-LOC
7,##AP,2.0,B-LOC,IGNORE,I-LOC
8,##AN,2.0,B-LOC,IGNORE,I-LOC
9,GE,3.0,O,O,O


IGNORE positions: 21 / 33 tokens in this sentence
IGNORE index = -100


## 4. Experiment 1 — thống kê trên toàn test set

Yêu cầu của W9 là tối thiểu 20 câu; ở đây dùng toàn bộ 3.453 câu test.
`R = #subword / #word`.

In [5]:
stats = read_json("predictions/tokenization_analysis.json")
summary = pd.DataFrame([
    ("sentences", stats["n_sentences"]),
    ("words", stats["n_words"]),
    ("subword tokens (no specials)", stats["n_subwords"]),
    ("subwords / word", round(stats["ratio_subwords_per_word"], 4)),
    ("words split into >1 piece", stats["n_split_words"]),
    ("split rate", round(stats["split_rate"], 4)),
    ("short sentences (<=15 words), ratio", round(stats["short_sentences"]["ratio"], 4)),
    ("long sentences (>15 words), ratio", round(stats["long_sentences"]["ratio"], 4)),
    ("split rate inside gold entities", round(stats["entity_words_split_rate"], 4)),
    ("split rate outside entities", round(stats["non_entity_words_split_rate"], 4)),
    ("max subwords in a sentence", stats["max_tokens"]),
    ("sentences longer than 128 subwords", stats["sentences_over_128_subwords"]),
], columns=["metric", "value"])
display(summary)

,metric,value
0,sentences,3453.0000
1,words,46435.0000
2,subword tokens (no specials),63461.0000
3,subwords / word,1.3667
4,words split into >1 piece,8287.0000
5,split rate,0.1785
6,"short sentences (<=15 words), ratio",1.6960
7,"long sentences (>15 words), ratio",1.1910
8,split rate inside gold entities,0.4528
9,split rate outside entities,0.1204


In [6]:
sentence_stats = pd.read_csv("tokenization_stats.csv")
sentence_stats["length_bucket"] = pd.cut(
    sentence_stats["n_words"], bins=[0, 15, 1000], labels=["short (<=15)", "long (>15)"]
)
display(sentence_stats.groupby("length_bucket", observed=True).agg(
    sentences=("id", "count"),
    mean_subwords_per_word=("ratio", "mean"),
    split_words=("n_split_words", "sum"),
).round(4))
print("Words split most often:")
display(pd.DataFrame(stats["most_split_words"], columns=["word", "sentences"]))
print("Most common continuation pieces:")
display(pd.DataFrame(stats["most_common_continuations"], columns=["##piece", "count"]))

,sentences,mean_subwords_per_word,split_words
length_bucket,,,
short (<=15),2374,1.9626,4928
long (>15),1079,1.1910,3359


Words split most often:


,word,sentences
0,'s,330
1,1996-12-06,165
2,--,58
3,U.S.,51
4,1996-12-07,51
5,SOCCER,42
6,NEW,37
7,DIVISION,35
8,LONDON,31
9,n't,28


Most common continuation pieces:


,##piece,count
0,-,1368
1,.,1108
2,##s,341
3,s,334
4,12,246
5,",",233
6,##S,179
7,06,170
8,##O,158
9,##ON,131


## 5. Nhận xét

- WordPiece hiếm khi chia các từ thông dụng; các từ dài, tên riêng, chữ viết hoa
  có thể bị chia nhiều hơn; cần kiểm tra bằng bảng thống kê thực tế.
- Từ bị chia nhiều mảnh không tự động là lỗi alignment; cần kiểm chứng word_ids.
- Đối chiếu tỷ lệ ngắn/dài và entity/non-entity trong bảng; không giả định các
  tỷ lệ bằng nhau hoặc suy ra nguyên nhân lỗi chỉ từ correlation.
- Notebook chọn **Strategy A**: mỗi word có đúng một label, không nhân bản
  nhãn sang các mảnh `##`, giữ mỗi word một vị trí supervision.
  Continuation vẫn tham gia attention, chỉ không trực tiếp đóng góp loss.